In [ ]:
# Cell 1 — Install dependencies
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl

In [ ]:
# Cell 2 — Imports + GPU check
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

assert torch.cuda.is_available(), "No GPU found! Change runtime to GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Config

MODEL_ID    = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_ROWS    = 10000
MAX_SEQ_LEN = 512
OUTPUT_DIR  = "./qlora-mistral-output"

In [ ]:
# Cell 4 — Load model in 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — better than fp4 for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # quantize the quantization constants too
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded successfully")

In [ ]:
# Cell 5 — LoRA adapters
lora_config = LoraConfig(
    r=16,                   # rank — higher = more params, better fit, more VRAM
    lora_alpha=32,          # scaling factor, keep at 2 × r
    target_modules=[        # inject adapters into all linear layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.4% trainable params (~27M of 7B total)

In [ ]:
# Cell 6 — Load + format dataset

def format_germanquad(row):
    question = row["question"]
    answer = row["answers"]["text"][0]
    row["text"] = (
        f"<s>[INST] Beantworte die folgende Frage auf Deutsch:\n\n"
        f"{question} [/INST] {answer} </s>"
    )
    return row

dataset = (
    load_dataset("parquet",
                 data_files="hf://datasets/deepset/germanquad@~parquet/plain_text/train/0000.parquet",
                 split="train")
    .map(format_germanquad)
    .select(range(min(MAX_ROWS, 13722)))
)
print(f"Sample: {dataset[0]['text'][:300]}")
print(f"Dataset size: {len(dataset)} rows")

In [ ]:
# Cell 7 — Train
results = {}
for lr in [5e-5, 1e-4, 2e-4, 5e-4, 1e-3]:
    args = SFTConfig(
        output_dir=f"{OUTPUT_DIR}/lr-search-{lr}",
        max_steps=50,                    # just 50 steps, not a full epoch
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=lr,
        fp16=False,
        logging_steps=10,
        max_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        packing=False,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=args,
        processing_class=tokenizer,
    )
    trainer.train()

    final_loss = [x["loss"] for x in trainer.state.log_history if "loss" in x][-1]
    results[lr] = final_loss
    print(f"LR {lr:.0e} → loss {final_loss:.4f}")

print("\n--- Results ---")
for lr, loss in sorted(results.items(), key=lambda x: x[1]):
    print(f"LR {lr:.0e} → {loss:.4f}")